<a href="https://colab.research.google.com/github/chk-33/large-language-modelloing/blob/main/qwen%20fine%20tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q -U bitsandbytes

import torch
import bitsandbytes
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from transformers.utils import is_bitsandbytes_available

print("bitsandbytes:", bitsandbytes.__version__)
print("Detected by Transformers:", is_bitsandbytes_available())
print("GPU:", torch.cuda.get_device_name(0))

assert is_bitsandbytes_available(), (
    "Transformers cannot detect bitsandbytes. Restart the Colab runtime."
)

model_name = "Qwen/Qwen2.5-0.5B"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto",
)

print("\nModel loaded successfully!")
print("Model class:", model.__class__.__name__)
print("4-bit enabled:", model.is_loaded_in_4bit)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.6 MB/s eta 0:00:00
bitsandbytes: 0.50.2
Detected by Transformers: True
GPU: Tesla T4


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]


Model loaded successfully!
Model class: Qwen2ForCausalLM
4-bit enabled: True


In [2]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

print("Base model prepared for QLoRA.")
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()
for name, param in model.named_parameters():
    if "lora_" in name:
        print(
            f"{name:75s} "
            f"shape={str(tuple(param.shape)):20s} "
            f"trainable={param.requires_grad}"
        )

Base model prepared for QLoRA.
trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184
base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight      shape=(8, 896)             trainable=True
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight      shape=(896, 8)             trainable=True
base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight      shape=(8, 896)             trainable=True
base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight      shape=(128, 8)             trainable=True
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight      shape=(8, 896)             trainable=True
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight      shape=(128, 8)             trainable=True
base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight      shape=(8, 896)             trainable=True
base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.we

In [3]:
texts = [
    "Artificial intelligence enables computers to perform tasks that usually require human intelligence.",
    "Machine learning allows models to learn patterns from data.",
    "Deep learning uses neural networks with multiple layers.",
    "Transformers use attention mechanisms to process sequences of tokens.",
    "Fine-tuning adapts a pretrained model to a specific task.",
    "QLoRA combines low-rank adapters with quantized model weights.",
    "Python is widely used for artificial intelligence and machine learning.",
    "A language model learns to predict the next token in a sequence.",
]

print("Number of examples:", len(texts))
print("\nFirst example:")
print(texts[0])
example = texts[0]

tokens = tokenizer(
    example,
    return_tensors="pt",
)

tokenized = tokenizer(
    texts,
    truncation=True,

    max_length=128,
    padding=False,
)
from transformers import DataCollatorForLanguageModeling
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    def __init__(self, tokenized_data):
        self.data = tokenized_data["input_ids"]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        return {"input_ids": self.data[index]}

dataset = TextDataset(tokenized)

collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collator,
)

batch = next(iter(loader))

for key, value in batch.items():
    print(key, value.shape)
optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad),
    lr=2e-4,
    weight_decay=0.01,
)
import torch

# Only optimize trainable LoRA parameters
trainable_params = [
    p for p in model.parameters()
    if p.requires_grad
]

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=2e-4,
    weight_decay=0.01,
)

print("Trainable tensors:", len(trainable_params))
print(
    "Trainable parameters:",
    sum(p.numel() for p in trainable_params)
)
from tqdm.auto import tqdm

model.train()

epochs = 3
max_steps = 30
step = 0

for epoch in range(epochs):
    progress = tqdm(loader, desc=f"Epoch {epoch + 1}")

    for batch in progress:
        batch = {
            key: value.to(model.device)
            for key, value in batch.items()
        }

        optimizer.zero_grad(set_to_none=True)

        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()

        # Prevent unusually large adapter updates
        torch.nn.utils.clip_grad_norm_(
            trainable_params,
            max_norm=1.0,
        )

        optimizer.step()

        step += 1
        progress.set_postfix(loss=f"{loss.item():.4f}")

        if step >= max_steps:
            break

    if step >= max_steps:
        break

print("Training finished.")
print("Final training loss:", f"{loss.item():.4f}")


Number of examples: 8

First example:
Artificial intelligence enables computers to perform tasks that usually require human intelligence.
input_ids torch.Size([2, 13])
attention_mask torch.Size([2, 13])
labels torch.Size([2, 13])
Trainable tensors: 192
Trainable parameters: 1081344


Epoch 1:   0%|          | 0/4 [00:00<?, ?it/s]

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch 2:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch 3:   0%|          | 0/4 [00:00<?, ?it/s]

Training finished.
Final training loss: 1.9353


In [4]:
%pip install -q -U bitsandbytes

import torch
import bitsandbytes
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from transformers.utils import is_bitsandbytes_available

print("bitsandbytes:", bitsandbytes.__version__)
print("Detected by Transformers:", is_bitsandbytes_available())
print("GPU:", torch.cuda.get_device_name(0))

assert is_bitsandbytes_available(), (
    "Transformers cannot detect bitsandbytes. Restart the Colab runtime."
)

model_name = "Qwen/Qwen2.5-0.5B"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto",
)

print("\nModel loaded successfully!")
print("Model class:", model.__class__.__name__)
print("4-bit enabled:", model.is_loaded_in_4bit)

bitsandbytes: 0.50.2
Detected by Transformers: True
GPU: Tesla T4


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


Model loaded successfully!
Model class: Qwen2ForCausalLM
4-bit enabled: True


In [5]:
texts = [
    "Artificial intelligence enables computers to perform tasks that usually require human intelligence.",
    "Machine learning allows models to learn patterns from data.",
    "Deep learning uses neural networks with multiple layers.",
    "Transformers use attention mechanisms to process sequences of tokens.",
    "Fine-tuning adapts a pretrained model to a specific task.",
    "QLoRA combines low-rank adapters with quantized model weights.",
    "Python is widely used for artificial intelligence and machine learning.",
    "A language model learns to predict the next token in a sequence.",
]

print("Number of examples:", len(texts))
print("\nFirst example:")
print(texts[0])
example = texts[0]

tokens = tokenizer(
    example,
    return_tensors="pt",
)

tokenized = tokenizer(
    texts,
    truncation=True,

    max_length=128,
    padding=False,
)
from transformers import DataCollatorForLanguageModeling
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    def __init__(self, tokenized_data):
        self.data = tokenized_data["input_ids"]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        return {"input_ids": self.data[index]}

dataset = TextDataset(tokenized)

collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collator,
)

batch = next(iter(loader))

for key, value in batch.items():
    print(key, value.shape)
optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad),
    lr=2e-4,
    weight_decay=0.01,
)
import torch

# Only optimize trainable LoRA parameters
trainable_params = [
    p for p in model.parameters()
    if p.requires_grad
]

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=2e-4,
    weight_decay=0.01,
)

print("Trainable tensors:", len(trainable_params))
print(
    "Trainable parameters:",
    sum(p.numel() for p in trainable_params)
)
from tqdm.auto import tqdm

model.train()

epochs = 3
max_steps = 30
step = 0

for epoch in range(epochs):
    progress = tqdm(loader, desc=f"Epoch {epoch + 1}")

    for batch in progress:
        batch = {
            key: value.to(model.device)
            for key, value in batch.items()
        }

        optimizer.zero_grad(set_to_none=True)

        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()

        # Prevent unusually large adapter updates
        torch.nn.utils.clip_grad_norm_(
            trainable_params,
            max_norm=1.0,
        )

        optimizer.step()

        step += 1
        progress.set_postfix(loss=f"{loss.item():.4f}")

        if step >= max_steps:
            break

    if step >= max_steps:
        break

print("Training finished.")
print("Final training loss:", f"{loss.item():.4f}")


Number of examples: 8

First example:
Artificial intelligence enables computers to perform tasks that usually require human intelligence.
input_ids torch.Size([2, 14])
attention_mask torch.Size([2, 14])
labels torch.Size([2, 14])
Trainable tensors: 122
Trainable parameters: 136206208


Epoch 1:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch 2:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch 3:   0%|          | 0/4 [00:00<?, ?it/s]

Training finished.
Final training loss: nan
